# 05 — Demostración de Reutilización del Modelo

**Colaborador responsable:** Eric Toporek

Este notebook demuestra que el modelo serializado puede cargarse y usarse para predicciones **sin re-entrenar**. Diseñado para la demostración en vivo de la presentación oral del 10 de junio.

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Añadir la raíz del proyecto al path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Semilla de reproducibilidad
from src.utils.constants import RANDOM_STATE

import pandas as pd
import numpy as np
import joblib

from src.utils.constants import MODELS_PATH, SEVERIDAD_MAP

## 2. Carga del modelo serializado

> **Importante:** Este notebook NO entrena ningún modelo. Solo carga el modelo previamente guardado en `models/`.

In [2]:
# Cargar todos los modelos y metadatos (descargándolos de Hugging Face si no existen localmente)
import joblib
import urllib.request
from pathlib import Path

# Asegurar que el directorio de modelos existe
MODELS_PATH.mkdir(parents=True, exist_ok=True)

# Leer la base URL de models/url.txt
url_file = MODELS_PATH / "url.txt"
with open(url_file, "r") as f:
    base_url = f.read().strip()

filenames = [
    "decision_tree.joblib",
    "random_forest.joblib",
    "xgboost_model.joblib",
    "features.joblib",
    "preprocessor.joblib"
]

# Descargar archivos faltantes
for filename in filenames:
    file_path = MODELS_PATH / filename
    if not file_path.exists():
        download_url = f"{base_url}{filename}"
        print(f"Descargando {filename} desde Hugging Face...")
        urllib.request.urlretrieve(download_url, file_path)
        print(f"  Descargado con éxito en: {file_path}")

# Cargar archivos
modelo_dt = joblib.load(MODELS_PATH / "decision_tree.joblib")
modelo_rf = joblib.load(MODELS_PATH / "random_forest.joblib")
modelo_xgb = joblib.load(MODELS_PATH / "xgboost_model.joblib")
features = joblib.load(MODELS_PATH / "features.joblib")
preprocessor = joblib.load(MODELS_PATH / "preprocessor.joblib")

print("Modelos cargados correctamente:")
print(f"  - Decision Tree : {type(modelo_dt).__name__}")
print(f"  - Random Forest : {type(modelo_rf).__name__}")
print(f"  - XGBoost       : {type(modelo_xgb).__name__}")
print(f"Número de features esperadas: {len(features)}")

Modelos cargados correctamente:
  - Decision Tree : DecisionTreeClassifier
  - Random Forest : RandomForestClassifier
  - XGBoost       : XGBClassifier
Número de features esperadas: 30


## 3. Datos de prueba

Simular un paciente nuevo con datos realistas.

In [3]:
# Definir dos perfiles de pacientes diferentes para la demostración
# Paciente 1: Persona mayor de edad con múltiples comorbilidades y neumonía
# Paciente 2: Persona joven sin comorbilidades y síntomas leves

# Definimos los valores crudos para los pacientes
pacientes_crudos = pd.DataFrame([
    {
        "EDAD": 78,
        "SEXO": 1,           # Hombre (codificado post-encoding)
        "DIABETES": 1,       # Sí
        "HIPERTENSION": 1,   # Sí
        "OBESIDAD": 1,       # Sí
        "EPOC": 0,           # No
        "ASMA": 0,           # No
        "INMUSUPR": 0,       # No
        "CARDIOVASCULAR": 1, # Sí
        "RENAL_CRONICA": 0,  # No
        "TABAQUISMO": 0,     # No
        "OTRA_COM": 0,       # No (comorbilidad añadida)
        "NEUMONIA": 1,       # Sí
        "EMBARAZO": 0,       # No aplica / No
        "HABLA_LENGUA_INDIG": 0,
        "INDIGENA": 0,
        "MIGRANTE": 0,
        "OTRO_CASO": 1,      # Sí
        "NACIONALIDAD": 1,   # Mexicano
        "SECTOR": 4,         # IMSS
        "ENTIDAD_NAC": 9,    # CDMX (añadido)
        "ENTIDAD_UM": 9,     # CDMX
        "ENTIDAD_RES": 9,
        "MUNICIPIO_RES": 2,
        "TOMA_MUESTRA_LAB": 1,
        "TOMA_MUESTRA_ANTIGENO": 0,
        "CLASIFICACION_FINAL_COVID": 3, # Confirmado
        "CLASIFICACION_FINAL_FLU": 5,   # Negativo
        "RESULTADO_PCR": 1,
        "RESULTADO_PCR_COINFECCION": 0
    },
    {
        "EDAD": 25,
        "SEXO": 0,           # Mujer
        "DIABETES": 0,       # No
        "HIPERTENSION": 0,   # No
        "OBESIDAD": 0,       # No
        "EPOC": 0,
        "ASMA": 0,
        "INMUSUPR": 0,
        "CARDIOVASCULAR": 0,
        "RENAL_CRONICA": 0,
        "TABAQUISMO": 1,     # Sí
        "OTRA_COM": 0,       # No (comorbilidad añadida)
        "NEUMONIA": 0,       # No
        "EMBARAZO": 0,       # No
        "HABLA_LENGUA_INDIG": 0,
        "INDIGENA": 0,
        "MIGRANTE": 0,
        "OTRO_CASO": 1,
        "NACIONALIDAD": 1,
        "SECTOR": 12,        # SSA
        "ENTIDAD_NAC": 15,   # EdoMex (añadido)
        "ENTIDAD_UM": 15,    # EdoMex
        "ENTIDAD_RES": 15,
        "MUNICIPIO_RES": 12,
        "TOMA_MUESTRA_LAB": 0,
        "TOMA_MUESTRA_ANTIGENO": 1,
        "CLASIFICACION_FINAL_COVID": 5, # Negativo
        "CLASIFICACION_FINAL_FLU": 5,
        "RESULTADO_PCR": 0,
        "RESULTADO_PCR_COINFECCION": 0
    }
])

# Aplicamos el mismo escalador de EDAD entrenado para obtener EDAD_SCALED
scaler = preprocessor._edad_scaler
pacientes_crudos["EDAD_SCALED"] = scaler.transform(pacientes_crudos[["EDAD"]]).flatten()

# Seleccionar solo las columnas ordenadas que corresponden a las features del modelo
pacientes_demo = pacientes_crudos[features]
print("Datos de pacientes procesados y listos para inferencia:")
pacientes_demo

Datos de pacientes procesados y listos para inferencia:


,SECTOR,ENTIDAD_UM,SEXO,ENTIDAD_NAC,ENTIDAD_RES,MUNICIPIO_RES,NEUMONIA,NACIONALIDAD,EMBARAZO,HABLA_LENGUA_INDIG,...,TABAQUISMO,OTRO_CASO,TOMA_MUESTRA_LAB,RESULTADO_PCR,RESULTADO_PCR_COINFECCION,TOMA_MUESTRA_ANTIGENO,CLASIFICACION_FINAL_COVID,CLASIFICACION_FINAL_FLU,MIGRANTE,EDAD_SCALED
0,4,9,1,9,9,2,1,1,0,0,...,0,1,1,1,0,0,3,5,0,1.622232
1,12,15,0,15,15,12,0,1,0,0,...,1,1,0,0,0,1,5,5,0,-0.382001


## 4. Predicción

In [4]:
# Ejecutar predicciones con todos los modelos para comparar comportamientos
modelos_cargados = {
    "Decision Tree": modelo_dt,
    "Random Forest": modelo_rf,
    "XGBoost": modelo_xgb
}

for idx, label in [(0, "Paciente 1 (Alto Riesgo: 78 años, Diabetes/Hipertensión, Neumonía)"),
                    (1, "Paciente 2 (Bajo Riesgo: 25 años, Fumador, Sin comorbilidades)")]:
    print("=" * 80)
    print(f" PREDICCIONES PARA: {label}")
    print("=" * 80)
    
    paciente_df = pacientes_demo.iloc[[idx]]
    
    # Creamos una tabla comparativa de predicciones
    for name, model in modelos_cargados.items():
        pred = model.predict(paciente_df)
        probs = model.predict_proba(paciente_df)[0]
        
        nivel = int(pred[0])
        etiqueta = SEVERIDAD_MAP[nivel]
        
        print(f" Modelo: {name:<15} -> Predicción: {nivel} ({etiqueta})")
        print("   Probabilidades:")
        for i, (k, v) in enumerate(SEVERIDAD_MAP.items()):
            print(f"     {v:<22}: {probs[i]:.2%}")
        print("-" * 50)
    print()

 PREDICCIONES PARA: Paciente 1 (Alto Riesgo: 78 años, Diabetes/Hipertensión, Neumonía)
 Modelo: Decision Tree   -> Predicción: 1 (Grave (hospitalizado))
   Probabilidades:
     Leve (ambulatorio)    : 0.00%
     Grave (hospitalizado) : 65.40%
     Crítico (UCI/intubado): 0.00%
     Fallecido             : 34.60%
--------------------------------------------------
 Modelo: Random Forest   -> Predicción: 1 (Grave (hospitalizado))
   Probabilidades:
     Leve (ambulatorio)    : 2.88%
     Grave (hospitalizado) : 57.77%
     Crítico (UCI/intubado): 8.34%
     Fallecido             : 31.01%
--------------------------------------------------
 Modelo: XGBoost         -> Predicción: 3 (Fallecido)
   Probabilidades:
     Leve (ambulatorio)    : 0.54%
     Grave (hospitalizado) : 24.73%
     Crítico (UCI/intubado): 3.78%
     Fallecido             : 70.94%
--------------------------------------------------

 PREDICCIONES PARA: Paciente 2 (Bajo Riesgo: 25 años, Fumador, Sin comorbilidades)
 Modelo

## 5. Demostración Interactiva en Terminal

Para demostraciones interactivas que requieren la entrada del usuario en tiempo real (por ejemplo, durante la presentación o pruebas dinámicas), hemos creado un script CLI en la terminal. Esto evita tener celdas interactivas con `input()` que bloqueen la renderización estática del sitio web en Quarto.

### Instrucciones de Ejecución

Para iniciar la demostración interactiva en la consola, ejecute:
```bash
python src/scripts/predict_interactive.py
```

#### Parámetros Avanzados:
- Guardar las respuestas ingresadas en un archivo JSON payload:
  ```bash
  python src/scripts/predict_interactive.py --output data/paciente_personalizado.json
  ```
- Leer directamente un payload JSON para evaluar al paciente sin responder el cuestionario:
  ```bash
  python src/scripts/predict_interactive.py --input data/paciente_ejemplo.json
  ```

## 6. Predicción a partir de un Payload JSON (No interactivo)

A continuación se demuestra cómo el notebook puede cargar un payload JSON generado por el script interactivo y realizar la predicción de forma estática.

In [5]:
import json
from src.utils.constants import COLS_COMORBILIDADES

# 1. Cargar el payload JSON generado por el script o manualmente
json_path = project_root / "data/paciente_ejemplo.json"
with open(json_path, "r", encoding="utf-8") as f:
    paciente_json = json.load(f)

# 2. Convertir a DataFrame
df_json = pd.DataFrame([paciente_json])

# 3. Preprocesar datos (mapeos binarios y de sexo)
binary_si_no = COLS_COMORBILIDADES + [
    "NEUMONIA", "TOMA_MUESTRA_LAB", "EMBARAZO", "HABLA_LENGUA_INDIG",
    "INDIGENA", "MIGRANTE", "OTRO_CASO", "NACIONALIDAD"
]
for col in binary_si_no:
    if col in df_json.columns:
        df_json[col] = df_json[col].map({1: 1, 2: 0})
        
if "SEXO" in df_json.columns:
    df_json["SEXO"] = df_json["SEXO"].map({1: 0, 2: 1})

if "TOMA_MUESTRA_ANTIGENO" in df_json.columns:
    df_json["TOMA_MUESTRA_ANTIGENO"] = df_json["TOMA_MUESTRA_ANTIGENO"].map({1: 1, 2: 0})

# 4. Escalar la EDAD usando el escalador de la clase preprocessor
df_json["EDAD_SCALED"] = preprocessor._edad_scaler.transform(df_json[["EDAD"]]).flatten()

# 5. Filtrar y ordenar según las features esperadas por el modelo
df_json_demo = df_json[features]

# 6. Ejecutar predicciones con todos los modelos
print("========================================================================")
print(f" PREDICCIONES DESDE PAYLOAD JSON ({json_path.name})")
print("========================================================================")
for name, model in modelos_cargados.items():
    pred = model.predict(df_json_demo)[0]
    probs = model.predict_proba(df_json_demo)[0]
    print(f" Modelo: {name:<15} -> Predicción: {pred} ({SEVERIDAD_MAP[int(pred)]})")
    print("   Probabilidades:")
    for i, (k, v) in enumerate(SEVERIDAD_MAP.items()):
        print(f"     {v:<22}: {probs[i]:.2%}")
    print("-" * 50)

 PREDICCIONES DESDE PAYLOAD JSON (paciente_ejemplo.json)
 Modelo: Decision Tree   -> Predicción: 1 (Grave (hospitalizado))
   Probabilidades:
     Leve (ambulatorio)    : 0.00%
     Grave (hospitalizado) : 65.40%
     Crítico (UCI/intubado): 0.00%
     Fallecido             : 34.60%
--------------------------------------------------
 Modelo: Random Forest   -> Predicción: 1 (Grave (hospitalizado))
   Probabilidades:
     Leve (ambulatorio)    : 4.16%
     Grave (hospitalizado) : 59.29%
     Crítico (UCI/intubado): 7.58%
     Fallecido             : 28.97%
--------------------------------------------------
 Modelo: XGBoost         -> Predicción: 3 (Fallecido)
   Probabilidades:
     Leve (ambulatorio)    : 0.46%
     Grave (hospitalizado) : 22.77%
     Crítico (UCI/intubado): 2.21%
     Fallecido             : 74.55%
--------------------------------------------------
